# Task 2.6 – DBRepo API Reimplementation

This notebook reimplements the experiment data-loading step so that data are retrieved **exclusively from the DBRepo REST API** instead of local CSV files.

It satisfies Task 2.6 by using DBRepo REST API endpoints only, avoiding local CSV reads in the final pipeline, implementing robust error handling, and providing verification helpers.

The final working implementation loads the normalized DBRepo tables through the REST API and reconstructs the ML-ready feature table in Python. The DBRepo views were created and verified separately in the DBRepo views notebook.

> Important: The final experiment code must not call `pd.read_csv()` for input data. Any optional CSV comparison or verification helper is for development only and is not part of the final pipeline.

## 1. Imports and Configuration

This section imports the required Python libraries and defines the DBRepo configuration used by the notebook.

The configuration stores the DBRepo REST API base URL, the target database UUID, the API page size, and the ML feature view name. The page size is set to 1000 rows because the DBRepo API returns table data through paginated requests.

No credentials are stored in this configuration. The username and password are requested later at runtime.

In [1]:
import logging
import time
from getpass import getpass
from typing import Any, Dict, List, Optional

import pandas as pd
import requests
from requests.auth import HTTPBasicAuth

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

CONFIG = {
    "BASE_URL": "https://test.dbrepo.tuwien.ac.at/api/v1",
    "DATABASE_ID": "b23492bd-f66d-4663-a1f5-f296767dbbdc",
    "PAGE_SIZE": 1000,
    "ML_VIEW_NAME": "vw_accident_casualty_ml_features",
}

## 2. DBRepo API Loader

This section defines a reusable DBRepo API loader class for Task 2.6.

The loader replaces local CSV-based input loading with REST API requests to DBRepo. It first loads the database metadata to map table names to DBRepo table UUIDs. It then retrieves table rows through the paginated DBRepo data endpoint.

The loader includes robust error handling. It retries temporary API failures such as 429, 500, 502, 503, and 504, and raises clear errors if a table is missing, an API response is empty, or DBRepo returns an unexpected status code.

The main function used in the final workflow is `load_relation()`, which loads a complete DBRepo table into a pandas dataframe. The optional `load_ml_features()` helper can be used if the ML-ready DBRepo view is available through the same API loading mechanism.

In [2]:
class DBRepoAPIError(RuntimeError):
    """Raised when the DBRepo API returns an unexpected response."""


class DBRepoExperimentLoader:
    """
    Loads experiment data exclusively from the DBRepo REST API.

    This class replaces all local CSV-based input loading for Task 2.6.
    """

    def __init__(
        self,
        base_url: str,
        database_id: str,
        username: Optional[str] = None,
        password: Optional[str] = None,
        page_size: int = 1000,
        timeout: int = 15,
    ):
        self.base_url = base_url.rstrip("/")
        self.database_id = database_id
        self.page_size = max(1, min(int(page_size), 1000))
        self.timeout = timeout

        self.session = requests.Session()
        if username and password:
            self.session.auth = HTTPBasicAuth(username, password)

        self.table_map = self._load_table_map()

    def _request_json(self, method: str, url: str, retries: int = 3, **kwargs) -> Any:
        """Execute a DBRepo API request with retries and robust error handling."""
        last_error = None

        for attempt in range(retries):
            try:
                response = self.session.request(
                    method=method,
                    url=url,
                    timeout=self.timeout,
                    headers={"Accept": "application/json"},
                    **kwargs,
                )

                if response.status_code in [200, 201, 202]:
                    if not response.text.strip():
                        return {}
                    return response.json()

                if response.status_code in [429, 500, 502, 503, 504]:
                    wait_seconds = 0.5 * (2 ** attempt)
                    logger.warning(
                        "Temporary DBRepo error %s for %s. Retrying in %.1fs.",
                        response.status_code,
                        url,
                        wait_seconds,
                    )
                    time.sleep(wait_seconds)
                    continue

                raise DBRepoAPIError(
                    f"Unexpected DBRepo response {response.status_code} for {url}: {response.text}"
                )

            except requests.exceptions.RequestException as exc:
                last_error = exc
                wait_seconds = 0.5 * (2 ** attempt)
                logger.warning(
                    "Connection error while requesting %s: %s. Retrying in %.1fs.",
                    url,
                    exc,
                    wait_seconds,
                )
                time.sleep(wait_seconds)

        raise DBRepoAPIError(
            f"DBRepo request failed after {retries} attempts for {url}: {last_error}"
        )

    def _load_table_map(self) -> Dict[str, str]:
        """Load DBRepo table names and UUIDs from the database metadata endpoint."""
        url = f"{self.base_url}/database/{self.database_id}"
        payload = self._request_json("GET", url)

        tables = payload.get("tables", [])
        if not tables:
            raise DBRepoAPIError("No tables found in DBRepo database metadata response.")

        table_map = {table["name"]: table["id"] for table in tables}
        logger.info("Loaded DBRepo tables: %s", list(table_map.keys()))

        return table_map

    @staticmethod
    def _extract_rows(payload: Any) -> List[Dict[str, Any]]:
        """Extract row data from common DBRepo response shapes."""
        if isinstance(payload, list):
            return payload

        if isinstance(payload, dict):
            for key in ["content", "data", "items", "records"]:
                value = payload.get(key)
                if isinstance(value, list):
                    return value

        return []

    def load_relation(self, relation_name: str) -> pd.DataFrame:
        """Load all rows from a DBRepo table or view using pagination."""
        relation_id = self.table_map.get(relation_name)

        if not relation_id:
            available = ", ".join(sorted(self.table_map.keys()))
            raise DBRepoAPIError(
                f"Relation '{relation_name}' was not found in DBRepo. "
                f"Available relations are: {available}"
            )

        all_rows: List[Dict[str, Any]] = []
        page = 0

        while True:
            url = (
                f"{self.base_url}/database/{self.database_id}"
                f"/table/{relation_id}/data?page={page}&size={self.page_size}"
            )

            payload = self._request_json("GET", url)
            rows = self._extract_rows(payload)

            if not rows:
                break

            all_rows.extend(rows)
            logger.info("Fetched page %s from '%s' with %s rows.", page, relation_name, len(rows))

            if len(rows) < self.page_size:
                break

            page += 1

        df = pd.DataFrame(all_rows)

        if df.empty:
            raise DBRepoAPIError(f"Relation '{relation_name}' returned no rows.")

        logger.info(
            "Loaded %s rows and %s columns from DBRepo relation '%s'.",
            len(df),
            len(df.columns),
            relation_name,
        )

        return df

    def load_ml_features(self, view_name: str = "vw_accident_casualty_ml_features") -> pd.DataFrame:
        """Load the ML-ready feature table from a DBRepo view."""
        df = self.load_relation(view_name)

        required_columns = [
            "reference_number",
            "person_id",
            "number_of_vehicles",
            "date",
            "time",
            "road_class_id",
            "road_surface_id",
            "lighting_id",
            "weather_id",
            "sex_id",
            "age",
            "casualty_class_id",
            "casualty_severity_id",
            "vehicle_id",
        ]

        missing = [col for col in required_columns if col not in df.columns]
        if missing:
            raise DBRepoAPIError(
                f"DBRepo view '{view_name}' is missing required columns: {missing}"
            )

        return df


## 3. Load Data from DBRepo API Only

This section connects to DBRepo using credentials entered at runtime and creates an instance of the `DBRepoExperimentLoader`.

The notebook then loads all required normalized tables directly from the DBRepo REST API. No local CSV files are used in this step.

The loaded tables include the main fact tables:

* `accident`
* `person`
* `accident_casualty`

and the lookup tables:

* `road_class`
* `road_surface`
* `lighting`
* `weather`
* `sex`
* `casualty_class`
* `casualty_severity`
* `vehicle`

These tables are later joined in Python to reconstruct the ML-ready feature table.

In [3]:
username = input("DBRepo username: ").strip()
password = getpass("DBRepo password: ")

api_loader = DBRepoExperimentLoader(
    base_url=CONFIG["BASE_URL"],
    database_id=CONFIG["DATABASE_ID"],
    username=username,
    password=password,
    page_size=CONFIG["PAGE_SIZE"],
)

# Load normalized tables from DBRepo REST API
accident_df = api_loader.load_relation("accident")
person_df = api_loader.load_relation("person")
accident_casualty_df = api_loader.load_relation("accident_casualty")

road_class_df = api_loader.load_relation("road_class")
road_surface_df = api_loader.load_relation("road_surface")
lighting_df = api_loader.load_relation("lighting")
weather_df = api_loader.load_relation("weather")
sex_df = api_loader.load_relation("sex")
casualty_class_df = api_loader.load_relation("casualty_class")
casualty_severity_df = api_loader.load_relation("casualty_severity")
vehicle_df = api_loader.load_relation("vehicle")

logger.info("Loaded all normalized DBRepo tables through the REST API.")

INFO: Loaded DBRepo tables: ['accident_casualty', 'accident', 'person', 'sex', 'casualty_severity', 'casualty_class', 'road_class', 'road_surface', 'vehicle', 'lighting', 'weather']
INFO: Fetched page 0 from 'accident' with 1000 rows.
INFO: Fetched page 1 from 'accident' with 1000 rows.
INFO: Fetched page 2 from 'accident' with 1000 rows.
INFO: Fetched page 3 from 'accident' with 1000 rows.
INFO: Fetched page 4 from 'accident' with 1000 rows.
INFO: Fetched page 5 from 'accident' with 1000 rows.
INFO: Fetched page 6 from 'accident' with 1000 rows.
INFO: Fetched page 7 from 'accident' with 1000 rows.
INFO: Fetched page 8 from 'accident' with 1000 rows.
INFO: Fetched page 9 from 'accident' with 1000 rows.
INFO: Fetched page 10 from 'accident' with 118 rows.
INFO: Loaded 10118 rows and 10 columns from DBRepo relation 'accident'.
INFO: Fetched page 0 from 'person' with 1000 rows.
INFO: Fetched page 1 from 'person' with 1000 rows.
INFO: Fetched page 2 from 'person' with 1000 rows.
INFO: Fetc

## 4. Reconstruct ML-ready feature table from API-loaded tables

In [ ]:
def add_description(
    df: pd.DataFrame,
    lookup_df: pd.DataFrame,
    key_col: str,
    description_col: str,
) -> pd.DataFrame:
    """
    Join a lookup table and rename its description column.
    """
    lookup = lookup_df[[key_col, "description"]].copy()
    lookup = lookup.rename(columns={"description": description_col})
    return df.merge(lookup, on=key_col, how="left")


raw_df = (
    accident_casualty_df
    .merge(accident_df, on="reference_number", how="left")
    .merge(person_df, on="person_id", how="left")
)

raw_df = add_description(raw_df, road_class_df, "road_class_id", "road_class")
raw_df = add_description(raw_df, road_surface_df, "road_surface_id", "road_surface")
raw_df = add_description(raw_df, lighting_df, "lighting_id", "lighting")
raw_df = add_description(raw_df, weather_df, "weather_id", "weather")
raw_df = add_description(raw_df, sex_df, "sex_id", "sex")
raw_df = add_description(raw_df, casualty_class_df, "casualty_class_id", "casualty_class")
raw_df = add_description(raw_df, casualty_severity_df, "casualty_severity_id", "casualty_severity")
raw_df = add_description(raw_df, vehicle_df, "vehicle_id", "vehicle")

logger.info("Reconstructed ML-ready feature table from DBRepo API tables.")
logger.info("Rows loaded: %s", len(raw_df))
logger.info("Columns loaded: %s", list(raw_df.columns))

raw_df.head()

INFO: Reconstructed ML-ready feature table from DBRepo API tables.
INFO: Rows loaded: 27727
INFO: Columns loaded: ['casualty_class_id', 'casualty_severity_id', 'person_id', 'reference_number', 'vehicle_id', 'date', 'easting', 'lighting_id', 'northing', 'number_of_vehicles', 'road_class_id', 'road_surface_id', 'time', 'weather_id', 'age', 'sex_id', 'road_class', 'road_surface', 'lighting', 'weather', 'sex', 'casualty_class', 'casualty_severity', 'vehicle']


,casualty_class_id,casualty_severity_id,person_id,reference_number,vehicle_id,date,easting,lighting_id,northing,number_of_vehicles,...,age,sex_id,road_class,road_surface,lighting,weather,sex,casualty_class,casualty_severity,vehicle
0,1,3,4055,0,9,2010-01-01,430408,4,437054,2,...,18,1,Unclassified,Wet / Damp,Darkness: street lights present and lit,Snowing without high winds,Male,Driver or rider,Slight,Car
1,1,2,4052,0,9,2010-01-01,430408,4,437054,2,...,85,2,Unclassified,Wet / Damp,Darkness: street lights present and lit,Snowing without high winds,Female,Driver or rider,Serious,Car
2,1,3,4053,0,9,2010-01-01,430408,4,437054,2,...,18,1,Unclassified,Wet / Damp,Darkness: street lights present and lit,Snowing without high winds,Male,Driver or rider,Slight,Car
3,1,3,4042,0,3,2010-01-01,430408,4,437054,2,...,20,1,Unclassified,Wet / Damp,Darkness: street lights present and lit,Snowing without high winds,Male,Driver or rider,Slight,Motorcycle over 50cc and up to 125cc
4,1,3,4051,0,9,2010-01-01,430408,4,437054,2,...,18,1,Unclassified,Wet / Damp,Darkness: street lights present and lit,Snowing without high winds,Male,Driver or rider,Slight,Car
